In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

df=pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")


In [ ]:
# Task 1: Write your code here:
df_clean= df.drop(columns=['Order_ID'])
df_clean.isnull().sum()

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df_clean.isnull().sum()) #finding number of null values in each column
for col in ['Weather', 'Time_of_Day', 'Traffic_Level']:#making objective coloumn as unknown because these columns contirbute in the target predtiction
    df_clean[col] = df_clean[col].fillna('unknown')

df_clean['Courier_Experience_yrs']=df_clean['Courier_Experience_yrs'].fillna(0)
df_clean['Delivery_Time']=df_clean['Delivery_Time'].fillna(df['Delivery_Time'].mean()) # we cant loose data about the target so we replace it with mean of the target


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df[col])
df_clean.head()

In [ ]:
# Task 5: Write your code here:
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)  #spliting dataset into features X and target y
y = df_clean['Delivery_Time'].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits=5
losses=[]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model=RandomForestRegressor(n_estimators=200)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):# k-fold makes each fold validation for an iteration
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index] #each time loop iterate we have a new fold as validation
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train,y_train)
  y_pred= model.predict(X_test)


  mae = mean_absolute_error(y_test, y_pred)
  losses.append(mae)


avg_loss=np.mean(losses) # caluclating avg loss across the fold

print(avg_loss)



In [ ]:
# Task 1: Write your code here:
feature_cols=['Distance_km', 'Weather', 'Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test, y_pred.flatten(), alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual time ', fontsize=12)
plt.ylabel('Predicted time ($)', fontsize=12)
plt.title('Predicted vs Actual delvery time', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Task Bonus: Write your code here:
